In [1]:
import os
import re
import subprocess
import sys


def pip(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])


if "jax" in sys.modules:
    raise RuntimeError("Restart session, then run this cell first before importing jax.")

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ.pop("JAX_PLATFORMS", None)
os.environ.pop("PJRT_DEVICE", None)
os.environ.pop("CUDA_VISIBLE_DEVICES", None)

jax_extra = os.environ.get("JAX_CUDA_EXTRA", "cuda12")

out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=30)
smi = (out.stdout or "") + (out.stderr or "")
print("--- nvidia-smi -L ---")
print(smi)
print("returncode:", out.returncode)

has_gpu = re.search(r"GPU\s+\d+:", smi) is not None
bad = "libnvidia-ml" in smi.lower() or "couldn't find libnvidia" in smi.lower()
if not has_gpu and (bad or out.returncode != 0):
    raise RuntimeError("No GPU visible to this kernel. Runtime -> Change runtime type -> GPU.")

pip(["--upgrade", "pip"])
pip(["--upgrade", f"jax[{jax_extra}]", "optax", "torch", "torchvision", "torchaudio"])

print("Install done. Now go to Runtime -> Restart session, then run the remaining cells.")


--- nvidia-smi -L ---
GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-653b7edd-3008-e11a-1780-1f690341f0cd)

returncode: 0
Install done. Now go to Runtime -> Restart session, then run the remaining cells.


In [2]:
import torch
import jax
import jax.numpy as jnp

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("JAX", jax.__version__, jax.devices(), jax.default_backend())

x = jnp.ones((256, 256), jnp.float32)
jax.block_until_ready(x @ x)
print("JAX GPU OK")


torch 2.11.0+cu130 cuda True
JAX 0.9.2 [CudaDevice(id=0)] gpu
JAX GPU OK


In [3]:
import pathlib
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
path = pathlib.Path("input.txt")
if not path.exists():
    urllib.request.urlretrieve(url, path)
print(path.resolve(), path.stat().st_size)


/content/input.txt 1115394


In [4]:
%%writefile model.py
"""
Shared, parameterized GPT architecture for the research experiments.

This is the same decoder-only transformer used in all hand-designed models
(and in v2.py), but with hyperparameters passed as constructor arguments
instead of module-level globals.
"""

import torch
import torch.nn as nn
from torch.nn import functional as F


class Head(nn.Module):
    """One head of self-attention."""

    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""

    def __init__(self, n_embd, num_heads, head_size, block_size, dropout=0.0):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Simple linear → ReLU → linear → dropout."""

    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: LayerNorm → Attention → Residual → LayerNorm → FFN → Residual."""

    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_embd, n_head, head_size, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPT(nn.Module):
    """
    Tiny decoder-only GPT.

    Args:
        vocab_size: number of tokens
        n_embd:     embedding dimension
        n_head:     number of attention heads
        n_layer:    number of transformer blocks
        block_size: maximum sequence length
        dropout:    dropout rate (default 0.0)
    """

    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=True)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        device = idx.device
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            return logits, None

        B, T, C = logits.shape
        logits_flat = logits.view(B * T, C)
        targets_flat = targets.view(B * T)
        loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=-100)
        return logits, loss

    def get_attention_weights(self, idx):
        """
        Run a forward pass and return the attention weight matrices
        from every head in every layer.

        Returns:
            list of list of tensors: weights[layer][head] is (B, T, T)
        """
        B, T = idx.shape
        device = idx.device
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb

        all_weights = []
        for block in self.blocks:
            x_ln = block.ln1(x)
            layer_weights = []
            head_outputs = []
            for head in block.sa.heads:
                k = head.key(x_ln)
                q = head.query(x_ln)
                wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
                wei = wei.masked_fill(head.tril[:T, :T] == 0, float('-inf'))
                wei = F.softmax(wei, dim=-1)
                layer_weights.append(wei.detach())
                v = head.value(x_ln)
                head_outputs.append(wei @ v)
            all_weights.append(layer_weights)
            attn_out = torch.cat(head_outputs, dim=-1)
            attn_out = block.sa.dropout(block.sa.proj(attn_out))
            x = x + attn_out
            x = x + block.ffwd(block.ln2(x))

        return all_weights


Writing model.py


In [5]:
%%writefile jax_lm.py
"""
JAX implementation of the same decoder-only GPT as `model.py` (research).

Used for timing / throughput comparisons against PyTorch. Not wired into the
generalization experiments (those stay PyTorch + `model.py`).

Dependencies: pip install jax optax
  - GPU: follow https://github.com/jax-ml/jax (e.g. jax[cuda12] on Linux CUDA)
  - Apple Silicon: JAX often runs on CPU; see JAX install docs for experimental GPU.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Mapping, Tuple

import jax
import jax.numpy as jnp
import numpy as np
import optax


@dataclass(frozen=True)
class GPTConfig:
    vocab_size: int
    n_embd: int
    n_head: int
    n_layer: int
    block_size: int
    dropout: float = 0.0


def layer_norm(x: jnp.ndarray, scale: jnp.ndarray, bias: jnp.ndarray, eps: float = 1e-5) -> jnp.ndarray:
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / jnp.sqrt(var + eps)
    return scale * x_norm + bias


def dropout(rng: jax.Array, x: jnp.ndarray, rate: float, train: bool) -> jnp.ndarray:
    if not train or rate <= 0.0:
        return x
    keep = 1.0 - rate
    mask = jax.random.bernoulli(rng, p=keep, shape=x.shape)
    return jnp.where(mask, x / keep, 0.0)


def linear(x: jnp.ndarray, kernel: jnp.ndarray, bias: jnp.ndarray | None) -> jnp.ndarray:
    # kernel: (in_dim, out_dim)  ->  x @ kernel + bias
    y = jnp.dot(x, kernel)
    if bias is not None:
        y = y + bias
    return y


def causal_mask(T: int) -> jnp.ndarray:
    return jnp.tril(jnp.ones((T, T), dtype=jnp.float32))


def head_forward(
    rng: jax.Array,
    x: jnp.ndarray,
    Wq: jnp.ndarray,
    Wk: jnp.ndarray,
    Wv: jnp.ndarray,
    tril_T: jnp.ndarray,
    dropout_rate: float,
    train: bool,
) -> jnp.ndarray:
    """Single attention head. Shapes: W* are (n_embd, head_size). tril_T is (T, T)."""
    hs = Wq.shape[1]
    q = jnp.dot(x, Wq)
    k = jnp.dot(x, Wk)
    v = jnp.dot(x, Wv)
    wei = jnp.matmul(q, jnp.swapaxes(k, -2, -1)) * (hs ** -0.5)
    wei = jnp.where(tril_T[None, :, :] == 0, -1e9, wei)
    wei = jax.nn.softmax(wei, axis=-1)
    rng, dr = jax.random.split(rng)
    wei = dropout(dr, wei, dropout_rate, train)
    return jnp.matmul(wei, v)


def block_forward(
    rng: jax.Array,
    x: jnp.ndarray,
    block: Dict[str, Any],
    tril_T: jnp.ndarray,
    n_head: int,
    dropout_rate: float,
    train: bool,
) -> jnp.ndarray:
    ln1s, ln1b = block["ln1_scale"], block["ln1_bias"]
    ln2s, ln2b = block["ln2_scale"], block["ln2_bias"]

    h = layer_norm(x, ln1s, ln1b)
    head_outs = []
    for i in range(n_head):
        rng, hr = jax.random.split(rng)
        ho = head_forward(
            hr,
            h,
            block["Wq"][i],
            block["Wk"][i],
            block["Wv"][i],
            tril_T,
            dropout_rate,
            train,
        )
        head_outs.append(ho)
    sa = jnp.concatenate(head_outs, axis=-1)
    rng, dr = jax.random.split(rng)
    sa = linear(sa, block["proj_kernel"], block["proj_bias"])
    sa = dropout(dr, sa, dropout_rate, train)
    x = x + sa

    h2 = layer_norm(x, ln2s, ln2b)
    rng, dr = jax.random.split(rng)
    ff = linear(h2, block["ff1_kernel"], block["ff1_bias"])
    ff = jax.nn.relu(ff)
    ff = linear(ff, block["ff2_kernel"], block["ff2_bias"])
    ff = dropout(dr, ff, dropout_rate, train)
    x = x + ff
    return x


def gpt_forward(
    rng: jax.Array,
    params: Dict[str, Any],
    idx: jnp.ndarray,
    train: bool,
    cfg: GPTConfig,
) -> jnp.ndarray:
    """Returns logits (B, T, vocab). `params` is array-only (no config nested)."""
    B, T = idx.shape
    te = params["wte"][idx]
    pos = jnp.arange(T)[None, :]
    pe = params["wpe"][pos]
    x = te + pe
    tril_T = causal_mask(T)
    for i in range(cfg.n_layer):
        rng, lr = jax.random.split(rng)
        x = block_forward(lr, x, params["blocks"][i], tril_T, cfg.n_head, cfg.dropout, train)
    x = layer_norm(x, params["lnf_scale"], params["lnf_bias"])
    logits = linear(x, params["lm_head_kernel"], params["lm_head_bias"])
    return logits


def loss_fn(
    params: Dict[str, Any],
    rng: jax.Array,
    idx: jnp.ndarray,
    targets: jnp.ndarray,
    train: bool,
    cfg: GPTConfig,
) -> jnp.ndarray:
    logits = gpt_forward(rng, params, idx, train=train, cfg=cfg)
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    mask = targets != -100
    safe_targets = jnp.where(mask, targets, 0)
    token_loss = -jnp.take_along_axis(log_probs, safe_targets[..., None], axis=-1).squeeze(-1)
    return jnp.sum(token_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)


def init_params(key: jax.Array, cfg: GPTConfig) -> Dict[str, Any]:
    """Initialize parameters (similar scale to default PyTorch nn.Module init)."""
    keys = jax.random.split(key, 256)
    ki = 0
    scale = 0.02

    def randn(shape):
        nonlocal ki
        k = keys[ki]
        ki += 1
        return jax.random.normal(k, shape, dtype=jnp.float32) * scale

    d = cfg.n_embd
    hs = d // cfg.n_head
    T = cfg.block_size
    V = cfg.vocab_size

    wte = randn((V, d))
    wpe = randn((T, d))

    blocks = []
    for _ in range(cfg.n_layer):
        Wq = jnp.stack([randn((d, hs)) for _ in range(cfg.n_head)], axis=0)
        Wk = jnp.stack([randn((d, hs)) for _ in range(cfg.n_head)], axis=0)
        Wv = jnp.stack([randn((d, hs)) for _ in range(cfg.n_head)], axis=0)
        proj_kernel = randn((d, d))
        proj_bias = jnp.zeros((d,), dtype=jnp.float32)
        ff1_kernel = randn((d, 4 * d))
        ff1_bias = jnp.zeros((4 * d,), dtype=jnp.float32)
        ff2_kernel = randn((4 * d, d))
        ff2_bias = jnp.zeros((d,), dtype=jnp.float32)
        ln1_scale = jnp.ones((d,), dtype=jnp.float32)
        ln1_bias = jnp.zeros((d,), dtype=jnp.float32)
        ln2_scale = jnp.ones((d,), dtype=jnp.float32)
        ln2_bias = jnp.zeros((d,), dtype=jnp.float32)
        blocks.append(
            {
                "Wq": Wq,
                "Wk": Wk,
                "Wv": Wv,
                "proj_kernel": proj_kernel,
                "proj_bias": proj_bias,
                "ff1_kernel": ff1_kernel,
                "ff1_bias": ff1_bias,
                "ff2_kernel": ff2_kernel,
                "ff2_bias": ff2_bias,
                "ln1_scale": ln1_scale,
                "ln1_bias": ln1_bias,
                "ln2_scale": ln2_scale,
                "ln2_bias": ln2_bias,
            }
        )

    lnf_scale = jnp.ones((d,), dtype=jnp.float32)
    lnf_bias = jnp.zeros((d,), dtype=jnp.float32)
    lm_head_kernel = randn((d, V))
    lm_head_bias = jnp.zeros((V,), dtype=jnp.float32)

    return {
        "wte": wte,
        "wpe": wpe,
        "blocks": tuple(blocks),
        "lnf_scale": lnf_scale,
        "lnf_bias": lnf_bias,
        "lm_head_kernel": lm_head_kernel,
        "lm_head_bias": lm_head_bias,
    }


def pytorch_state_dict_to_jax_params(
    state_dict: Mapping[str, Any],
    cfg: GPTConfig,
) -> Dict[str, Any]:
    """
    Convert a PyTorch GPT `state_dict()` into the array layout used by this file.

    This lets PyTorch and JAX start from the exact same weights, which is the
    cleanest way to compare training behavior across frameworks.
    """

    def arr(name: str) -> jnp.ndarray:
        x = state_dict[name]
        if hasattr(x, "detach"):
            x = x.detach().cpu().numpy()
        else:
            x = np.asarray(x)
        return jnp.asarray(x, dtype=jnp.float32)

    blocks = []
    for layer in range(cfg.n_layer):
        Wq = jnp.stack(
            [arr(f"blocks.{layer}.sa.heads.{head}.query.weight").T for head in range(cfg.n_head)],
            axis=0,
        )
        Wk = jnp.stack(
            [arr(f"blocks.{layer}.sa.heads.{head}.key.weight").T for head in range(cfg.n_head)],
            axis=0,
        )
        Wv = jnp.stack(
            [arr(f"blocks.{layer}.sa.heads.{head}.value.weight").T for head in range(cfg.n_head)],
            axis=0,
        )
        blocks.append(
            {
                "Wq": Wq,
                "Wk": Wk,
                "Wv": Wv,
                "proj_kernel": arr(f"blocks.{layer}.sa.proj.weight").T,
                "proj_bias": arr(f"blocks.{layer}.sa.proj.bias"),
                "ff1_kernel": arr(f"blocks.{layer}.ffwd.net.0.weight").T,
                "ff1_bias": arr(f"blocks.{layer}.ffwd.net.0.bias"),
                "ff2_kernel": arr(f"blocks.{layer}.ffwd.net.2.weight").T,
                "ff2_bias": arr(f"blocks.{layer}.ffwd.net.2.bias"),
                "ln1_scale": arr(f"blocks.{layer}.ln1.weight"),
                "ln1_bias": arr(f"blocks.{layer}.ln1.bias"),
                "ln2_scale": arr(f"blocks.{layer}.ln2.weight"),
                "ln2_bias": arr(f"blocks.{layer}.ln2.bias"),
            }
        )

    return {
        "wte": arr("token_embedding_table.weight"),
        "wpe": arr("position_embedding_table.weight"),
        "blocks": tuple(blocks),
        "lnf_scale": arr("ln_f.weight"),
        "lnf_bias": arr("ln_f.bias"),
        "lm_head_kernel": arr("lm_head.weight").T,
        "lm_head_bias": arr("lm_head.bias"),
    }


def build_optimizer(
    learning_rate: float,
    *,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
    weight_decay: float = 0.0,
):
    return optax.adamw(
        learning_rate,
        b1=beta1,
        b2=beta2,
        eps=eps,
        weight_decay=weight_decay,
    )


def train_step_factory(
    cfg: GPTConfig,
    learning_rate: float,
    *,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
    weight_decay: float = 0.0,
):
    optimizer = build_optimizer(
        learning_rate,
        beta1=beta1,
        beta2=beta2,
        eps=eps,
        weight_decay=weight_decay,
    )

    def loss_train(p, r, xi, yi):
        return loss_fn(p, r, xi, yi, train=True, cfg=cfg)

    @jax.jit
    def _step(params, opt_state, rng, x, y):
        loss, grads = jax.value_and_grad(loss_train)(params, rng, x, y)
        updates, new_opt = optimizer.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt, loss

    return _step, optimizer


def numpy_batches(
    data: np.ndarray,
    batch_size: int,
    block_size: int,
    n_batches: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Precompute random batches shared by Torch/JAX for fair timing."""
    rng = np.random.RandomState(seed)
    max_i = len(data) - block_size
    xs = np.zeros((n_batches, batch_size, block_size), dtype=np.int64)
    ys = np.zeros((n_batches, batch_size, block_size), dtype=np.int64)
    for b in range(n_batches):
        ix = rng.randint(0, max_i, size=(batch_size,))
        for j, i in enumerate(ix):
            xs[b, j] = data[i : i + block_size]
            ys[b, j] = data[i + 1 : i + block_size + 1]
    return xs, ys


Writing jax_lm.py


In [6]:
%%writefile train_compare_shakespeare.py
#!/usr/bin/env python3
"""
Full Tiny Shakespeare training: PyTorch (`model.GPT`) vs JAX (`jax_lm`).

This script is meant to answer the fairness question first:
are the two framework runs actually starting from the same model, seeing the
same batches, and using the same optimizer settings?

Defaults therefore favor a controlled comparison:
  - same data split
  - same precomputed batch order
  - shared initial weights (PyTorch init exported into JAX)
  - explicit AdamW hyperparameters
  - dropout off by default, because different dropout RNG streams make
    "same training run" claims much weaker

Usage (from `myNanoGpt/research`, `comp560` env):

  python train_compare_shakespeare.py --data ../input.txt
  python train_compare_shakespeare.py --data ../input.txt --max-iters 500
  python train_compare_shakespeare.py --dropout 0.2 --weight-decay 0.01
  python train_compare_shakespeare.py --no-shared-init    # deliberately looser comparison
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

import numpy as np
import torch

SCRIPT_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPT_DIR))

from model import GPT  # noqa: E402


def default_data_path() -> Path:
    return SCRIPT_DIR.parent / "input.txt"


def load_encode(path: Path) -> tuple[np.ndarray, int]:
    if not path.exists():
        print(f"Missing: {path}")
        sys.exit(1)
    text = path.read_text(encoding="utf-8")
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    data = np.array([stoi[c] for c in text], dtype=np.int64)
    return data, len(chars)


def make_batches(
    data: np.ndarray,
    n_batches: int,
    batch_size: int,
    block_size: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.RandomState(seed)
    max_i = len(data) - block_size
    xs = np.zeros((n_batches, batch_size, block_size), dtype=np.int64)
    ys = np.zeros((n_batches, batch_size, block_size), dtype=np.int64)
    for b in range(n_batches):
        ix = rng.randint(0, max_i, size=(batch_size,))
        for j, i in enumerate(ix):
            xs[b, j] = data[i : i + block_size]
            ys[b, j] = data[i + 1 : i + block_size + 1]
    return xs, ys


def build_shared_torch_state(
    *,
    vocab_size: int,
    block_size: int,
    n_layer: int,
    n_head: int,
    n_embd: int,
    dropout: float,
    seed: int,
) -> dict[str, torch.Tensor]:
    """
    Build one canonical PyTorch initialization and reuse it for both frameworks.

    Using the same initial weights removes the largest hidden source of mismatch
    in PT-vs-JAX training comparisons.
    """
    torch.manual_seed(seed)
    model = GPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout)
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}


@torch.no_grad()
def torch_estimate_loss(
    model: GPT,
    train_x: np.ndarray,
    train_y: np.ndarray,
    val_x: np.ndarray,
    val_y: np.ndarray,
    device: torch.device,
    eval_iters: int,
) -> tuple[float, float]:
    model.eval()
    tl = torch.zeros(eval_iters, device=device)
    vl = torch.zeros(eval_iters, device=device)
    for k in range(eval_iters):
        x = torch.from_numpy(train_x[k]).to(device=device, dtype=torch.long)
        y = torch.from_numpy(train_y[k]).to(device=device, dtype=torch.long)
        _, loss = model(x, y)
        tl[k] = loss
    for k in range(eval_iters):
        x = torch.from_numpy(val_x[k]).to(device=device, dtype=torch.long)
        y = torch.from_numpy(val_y[k]).to(device=device, dtype=torch.long)
        _, loss = model(x, y)
        vl[k] = loss
    model.train()
    return float(tl.mean().item()), float(vl.mean().item())


def run_pytorch(
    *,
    train_xs: np.ndarray,
    train_ys: np.ndarray,
    train_eval_x: np.ndarray,
    train_eval_y: np.ndarray,
    val_eval_x: np.ndarray,
    val_eval_y: np.ndarray,
    vocab_size: int,
    batch_size: int,
    block_size: int,
    n_layer: int,
    n_head: int,
    n_embd: int,
    dropout: float,
    lr: float,
    beta1: float,
    beta2: float,
    eps: float,
    weight_decay: float,
    seed: int,
    max_iters: int,
    eval_interval: int,
    eval_iters: int,
    initial_state_dict: dict[str, torch.Tensor] | None,
) -> dict:
    device = (
        torch.device("mps")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
        else torch.device("cuda")
        if torch.cuda.is_available()
        else torch.device("cpu")
    )
    if initial_state_dict is None:
        torch.manual_seed(seed)
    model = GPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout)
    if initial_state_dict is not None:
        model.load_state_dict(initial_state_dict)
    model = model.to(device)
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        betas=(beta1, beta2),
        eps=eps,
        weight_decay=weight_decay,
    )

    init_train, init_val = torch_estimate_loss(
        model, train_eval_x, train_eval_y, val_eval_x, val_eval_y, device, eval_iters
    )

    t0 = time.perf_counter()
    last_train, last_val = float("nan"), float("nan")

    for it in range(max_iters):
        x = torch.from_numpy(train_xs[it]).to(device=device, dtype=torch.long)
        y = torch.from_numpy(train_ys[it]).to(device=device, dtype=torch.long)
        opt.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        opt.step()

        if it % eval_interval == 0 or it == max_iters - 1:
            tr, va = torch_estimate_loss(
                model, train_eval_x, train_eval_y, val_eval_x, val_eval_y, device, eval_iters
            )
            last_train, last_val = tr, va
            elapsed = time.perf_counter() - t0
            print(f"  [pt] step {it:5d} | train {tr:.4f} | val {va:.4f} | elapsed {elapsed:.1f}s", flush=True)
            if device.type == "cuda":
                torch.cuda.synchronize()
            elif device.type == "mps":
                torch.mps.synchronize()

    wall = time.perf_counter() - t0
    return {
        "device": str(device),
        "wall_s": wall,
        "initial_train": init_train,
        "initial_val": init_val,
        "final_train": last_train,
        "final_val": last_val,
    }


def run_jax(
    *,
    train_xs: np.ndarray,
    train_ys: np.ndarray,
    train_eval_x: np.ndarray,
    train_eval_y: np.ndarray,
    val_eval_x: np.ndarray,
    val_eval_y: np.ndarray,
    vocab_size: int,
    batch_size: int,
    block_size: int,
    n_layer: int,
    n_head: int,
    n_embd: int,
    dropout: float,
    lr: float,
    beta1: float,
    beta2: float,
    eps: float,
    weight_decay: float,
    seed: int,
    max_iters: int,
    eval_interval: int,
    eval_iters: int,
    initial_params: dict[str, object] | None,
) -> dict:
    import jax
    import jax.numpy as jnp
    from jax_lm import GPTConfig, init_params, loss_fn, train_step_factory

    cfg = GPTConfig(
        vocab_size=vocab_size,
        n_embd=n_embd,
        n_head=n_head,
        n_layer=n_layer,
        block_size=block_size,
        dropout=dropout,
    )
    if initial_params is None:
        key_init = jax.random.PRNGKey(seed)
        params = init_params(key_init, cfg)
    else:
        params = initial_params
    step, tx = train_step_factory(
        cfg,
        lr,
        beta1=beta1,
        beta2=beta2,
        eps=eps,
        weight_decay=weight_decay,
    )
    opt_state = tx.init(params)

    @jax.jit
    def eval_batch_loss(p, rng, xb, yb):
        return loss_fn(p, rng, xb, yb, train=False, cfg=cfg)

    def jax_mean_eval(p, key, x_stack: np.ndarray, y_stack: np.ndarray) -> float:
        s = 0.0
        k = key
        for i in range(eval_iters):
            k, sk = jax.random.split(k)
            xi = jnp.asarray(x_stack[i], dtype=jnp.int32)
            yi = jnp.asarray(y_stack[i], dtype=jnp.int32)
            li = eval_batch_loss(p, sk, xi, yi)
            jax.block_until_ready(li)
            s += float(li)
        return s / eval_iters

    init_train = jax_mean_eval(params, jax.random.PRNGKey(seed + 11), train_eval_x, train_eval_y)
    init_val = jax_mean_eval(params, jax.random.PRNGKey(seed + 12), val_eval_x, val_eval_y)

    t0 = time.perf_counter()
    last_train, last_val = float("nan"), float("nan")
    key_loop = jax.random.PRNGKey(seed)

    for it in range(max_iters):
        x = jnp.asarray(train_xs[it], dtype=jnp.int32)
        y = jnp.asarray(train_ys[it], dtype=jnp.int32)
        key_loop, k_step = jax.random.split(key_loop)
        params, opt_state, _ = step(params, opt_state, k_step, x, y)

        if it % eval_interval == 0 or it == max_iters - 1:
            key_loop, ke = jax.random.split(key_loop)
            tr = jax_mean_eval(params, ke, train_eval_x, train_eval_y)
            key_loop, ke2 = jax.random.split(key_loop)
            va = jax_mean_eval(params, ke2, val_eval_x, val_eval_y)
            last_train, last_val = tr, va
            elapsed = time.perf_counter() - t0
            print(f"  [jx] step {it:5d} | train {last_train:.4f} | val {last_val:.4f} | elapsed {elapsed:.1f}s", flush=True)

    wall = time.perf_counter() - t0
    devs = jax.devices()
    return {
        "device": str(devs[0]) if devs else "unknown",
        "wall_s": wall,
        "initial_train": init_train,
        "initial_val": init_val,
        "final_train": last_train,
        "final_val": last_val,
    }


def main() -> None:
    p = argparse.ArgumentParser(description="Full Shakespeare training: PyTorch vs JAX")
    p.add_argument("--data", type=Path, default=default_data_path())
    p.add_argument("--max-iters", type=int, default=5000, help="Optimizer steps (v2.py default 5000)")
    p.add_argument("--eval-interval", type=int, default=500)
    p.add_argument("--eval-iters", type=int, default=200, help="Batches to average for train/val loss (v2 uses 200)")
    p.add_argument("--batch-size", type=int, default=32)
    p.add_argument("--block-size", type=int, default=256)
    p.add_argument("--n-layer", type=int, default=6)
    p.add_argument("--n-head", type=int, default=6)
    p.add_argument("--n-embd", type=int, default=384)
    p.add_argument("--dropout", type=float, default=0.0)
    p.add_argument("--lr", type=float, default=3e-3)
    p.add_argument("--beta1", type=float, default=0.9)
    p.add_argument("--beta2", type=float, default=0.999)
    p.add_argument("--eps", type=float, default=1e-8)
    p.add_argument("--weight-decay", type=float, default=0.0)
    p.add_argument("--seed", type=int, default=1337)
    p.add_argument("--no-shared-init", action="store_true")
    p.add_argument("--torch-only", action="store_true")
    p.add_argument("--jax-only", action="store_true")
    args = p.parse_args()

    data, vocab_size = load_encode(args.data)
    n = int(0.9 * len(data))
    train_data = data[:n]
    val_data = data[n:]

    bs, T = args.batch_size, args.block_size
    ei = args.eval_iters
    seed = args.seed

    print("=== train_compare_shakespeare (fair cross-framework defaults) ===")
    print(f"data={args.data}  chars={vocab_size}  train_tokens={n}  val_tokens={len(val_data)}")
    print(
        f"batch={bs} block={T} layers={args.n_layer} heads={args.n_head} n_embd={args.n_embd} "
        f"dropout={args.dropout} lr={args.lr}"
    )
    print(
        f"max_iters={args.max_iters} eval_interval={args.eval_interval} eval_iters={ei} seed={seed}"
    )
    print(
        f"optimizer=AdamW beta1={args.beta1} beta2={args.beta2} eps={args.eps} "
        f"weight_decay={args.weight_decay}"
    )
    print(
        f"shared_init={'no' if args.no_shared_init else 'yes'}  "
        f"same_batch_order=yes"
    )
    if args.dropout != 0.0:
        print(
            "warning: dropout > 0 means the two runs will use different dropout masks, "
            "so they are no longer the exact same optimization path."
        )
    print("Precomputing training batches (same sequence for PT & JAX)...", flush=True)
    t_prep = time.perf_counter()
    train_xs, train_ys = make_batches(train_data, args.max_iters, bs, T, seed)
    train_eval_x, train_eval_y = make_batches(train_data, ei, bs, T, seed + 1)
    val_eval_x, val_eval_y = make_batches(val_data, ei, bs, T, seed + 2)
    print(f"  done in {time.perf_counter() - t_prep:.2f}s", flush=True)

    shared_state = None
    shared_jax_params = None
    if not args.no_shared_init:
        shared_state = build_shared_torch_state(
            vocab_size=vocab_size,
            block_size=T,
            n_layer=args.n_layer,
            n_head=args.n_head,
            n_embd=args.n_embd,
            dropout=args.dropout,
            seed=seed,
        )
        shared_param_count = sum(
            tensor.numel() for name, tensor in shared_state.items() if not name.endswith("tril")
        )
        print(f"Shared initialization built once: params={shared_param_count:,}", flush=True)
        if not args.torch_only:
            from jax_lm import GPTConfig, pytorch_state_dict_to_jax_params

            shared_cfg = GPTConfig(
                vocab_size=vocab_size,
                n_embd=args.n_embd,
                n_head=args.n_head,
                n_layer=args.n_layer,
                block_size=T,
                dropout=args.dropout,
            )
            shared_jax_params = pytorch_state_dict_to_jax_params(shared_state, shared_cfg)

    rp = None
    if not args.jax_only:
        print("\n[PyTorch]", flush=True)
        rp = run_pytorch(
            train_xs=train_xs,
            train_ys=train_ys,
            train_eval_x=train_eval_x,
            train_eval_y=train_eval_y,
            val_eval_x=val_eval_x,
            val_eval_y=val_eval_y,
            vocab_size=vocab_size,
            batch_size=bs,
            block_size=T,
            n_layer=args.n_layer,
            n_head=args.n_head,
            n_embd=args.n_embd,
            dropout=args.dropout,
            lr=args.lr,
            beta1=args.beta1,
            beta2=args.beta2,
            eps=args.eps,
            weight_decay=args.weight_decay,
            seed=seed,
            max_iters=args.max_iters,
            eval_interval=args.eval_interval,
            eval_iters=ei,
            initial_state_dict=shared_state,
        )
        print(
            f"  initial train/val={rp['initial_train']:.4f} / {rp['initial_val']:.4f}"
        )
        print(
            f"  device={rp['device']}  wall_s={rp['wall_s']:.1f}  "
            f"final train/val={rp['final_train']:.4f} / {rp['final_val']:.4f}"
        )

    rj = None
    if not args.torch_only:
        print("\n[JAX]", flush=True)
        rj = run_jax(
            train_xs=train_xs,
            train_ys=train_ys,
            train_eval_x=train_eval_x,
            train_eval_y=train_eval_y,
            val_eval_x=val_eval_x,
            val_eval_y=val_eval_y,
            vocab_size=vocab_size,
            batch_size=bs,
            block_size=T,
            n_layer=args.n_layer,
            n_head=args.n_head,
            n_embd=args.n_embd,
            dropout=args.dropout,
            lr=args.lr,
            beta1=args.beta1,
            beta2=args.beta2,
            eps=args.eps,
            weight_decay=args.weight_decay,
            seed=seed,
            max_iters=args.max_iters,
            eval_interval=args.eval_interval,
            eval_iters=ei,
            initial_params=shared_jax_params,
        )
        print(
            f"  initial train/val={rj['initial_train']:.4f} / {rj['initial_val']:.4f}"
        )
        print(
            f"  device={rj['device']}  wall_s={rj['wall_s']:.1f}  "
            f"final train/val={rj['final_train']:.4f} / {rj['final_val']:.4f}"
        )

    if rp is not None and rj is not None:
        train_gap = abs(rp["initial_train"] - rj["initial_train"])
        val_gap = abs(rp["initial_val"] - rj["initial_val"])
        print("\n[Fairness check]")
        print(f"  initial train loss gap: {train_gap:.6e}")
        print(f"  initial val loss gap:   {val_gap:.6e}")
        if not args.no_shared_init and args.dropout == 0.0:
            print("  expectation: these gaps should be near zero; otherwise the setups still differ.")
        print(f"  JAX/PyTorch wall-time speedup: {rp['wall_s'] / rj['wall_s']:.2f}x")

    print("\nDone. Compare wall_s only when both use comparable hardware (e.g. both GPU).")


if __name__ == "__main__":
    main()


Writing train_compare_shakespeare.py


In [10]:
!python -u train_compare_shakespeare.py \
  --data input.txt \
  --batch-size 64 \
  --block-size 256 \
  --n-layer 8 \
  --n-head 8 \
  --n-embd 512 \
  --max-iters 5000 \
  --eval-interval 100 \
  --eval-iters 20 \
  --dropout 0.0 \
  --weight-decay 0.0


=== train_compare_shakespeare (fair cross-framework defaults) ===
data=input.txt  chars=65  train_tokens=1003854  val_tokens=111540
batch=64 block=256 layers=8 heads=8 n_embd=512 dropout=0.0 lr=0.003
max_iters=5000 eval_interval=100 eval_iters=20 seed=1337
optimizer=AdamW beta1=0.9 beta2=0.999 eps=1e-08 weight_decay=0.0
shared_init=yes  same_batch_order=yes
Precomputing training batches (same sequence for PT & JAX)...
  done in 1.25s
Shared initialization built once: params=25,405,505

[PyTorch]
  [pt] step     0 | train 5.8901 | val 5.9492 | elapsed 3.1s
  [pt] step   100 | train 2.6955 | val 2.6897 | elapsed 27.8s
  [pt] step   200 | train 2.5557 | val 2.5463 | elapsed 52.5s
  [pt] step   300 | train 2.5129 | val 2.5227 | elapsed 77.2s
  [pt] step   400 | train 2.5015 | val 2.5088 | elapsed 101.8s
  [pt] step   500 | train 2.5005 | val 2.5141 | elapsed 126.5s
  [pt] step   600 | train 2.5098 | val 2.5212 | elapsed 151.1s
  [pt] step   700 | train 2.5602 | val 2.5762 | elapsed 175.8s
